# B3b Defense – 08 Naive Random Walk Baseline

**Objective:** Benchmark the XGBoost FDEFX model against a naive random-walk baseline
(carry-forward: $\hat{y}_t = y_{t-1}$) using the same `TimeSeriesSplit(n_splits=5)` protocol
as NB04. Reports per-fold MAE, RMSE, sMAPE, WMAPE, and MASE (XGBoost MAE divided by
in-sample one-step naive MAE). A compact OOS comparison for Q1 2026 is appended.

All MAE / RMSE figures are reported in **monthly USD billions** (SAAR values ÷ 12).
sMAPE and WMAPE are scale-free.

In [1]:
import os
import warnings
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings('ignore')

In [2]:
DATA_PROCESSED = '../data/processed/'
MODELS_DIR     = '../models/'

# Load feature matrix — explicit to_datetime() to avoid slow dateutil fallback in pandas 2.x
df = pd.read_csv(DATA_PROCESSED + 'defense_feature_matrix.csv', index_col='date')
df.index = pd.to_datetime(df.index)
df = df.sort_index()

feature_cols = [col for col in df.columns if col != 'FDEFX']
target_col   = 'FDEFX'

X = df[feature_cols]
y = df[target_col]

print(f'Feature matrix: {df.shape}  |  Date range: {df.index[0].date()} to {df.index[-1].date()}')
print(f'Target: {target_col}  |  Features: {len(feature_cols)}')

Feature matrix: (287, 33)  |  Date range: 2002-02-01 to 2025-12-01
Target: FDEFX  |  Features: 32


## Metric Definitions

Identical to NB04 — defined here to keep this notebook self-contained.

In [3]:
def smape(actual, pred):
    """Symmetric Mean Absolute Percentage Error (in %)."""
    numerator   = 2 * np.abs(actual - pred)
    denominator = np.abs(actual) + np.abs(pred)
    mask = denominator != 0
    return np.mean(numerator[mask] / denominator[mask]) * 100


def wmape(actual, pred):
    """Weighted Mean Absolute Percentage Error (in %)."""
    return np.sum(np.abs(actual - pred)) / np.sum(np.abs(actual)) * 100

## Step 1 – XGBoost CV Reproduction

Re-runs the NB04 training loop with the identical hyperparameters and CV split to obtain
fold-level metrics for comparison. Results are checked against Table 7.2 (monthly equivalent).

In [4]:
tscv = TimeSeriesSplit(n_splits=5)

xgb_results = []

for fold_idx, (train_idx, test_idx) in enumerate(tscv.split(X), start=1):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    model = xgb.XGBRegressor(
        n_estimators     = 200,
        max_depth        = 4,
        learning_rate    = 0.05,
        subsample        = 0.8,
        colsample_bytree = 0.6,
        reg_alpha        = 0.1,
        random_state     = 42,
        verbosity        = 0
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    xgb_results.append({
        'fold':       fold_idx,
        'train_idx':  train_idx,
        'test_idx':   test_idx,
        'y_test':     y_test.values,
        'y_pred_xgb': y_pred,
        'y_train':    y_train.values,
        'X_train':    X_train,
        'MAE_saar':   mean_absolute_error(y_test, y_pred),
        'RMSE_saar':  np.sqrt(mean_squared_error(y_test, y_pred)),
        'sMAPE':      smape(y_test.values, y_pred),
        'WMAPE':      wmape(y_test.values, y_pred),
    })

print('XGBoost CV complete. Verification (monthly equiv. = SAAR / 12):')
print(f'{"Fold":>6}  {"MAE_mo":>8}  {"sMAPE%":>8}')
for r in xgb_results:
    print(f'{r["fold"]:>6}  {r["MAE_saar"]/12:>8.1f}  {r["sMAPE"]:>8.2f}')
mean_mae_saar  = np.mean([r['MAE_saar']  for r in xgb_results])
mean_smape     = np.mean([r['sMAPE']     for r in xgb_results])
# WMAPE mean: recompute as pooled across all folds
all_y_test  = np.concatenate([r['y_test']     for r in xgb_results])
all_y_pred  = np.concatenate([r['y_pred_xgb'] for r in xgb_results])
mean_wmape  = wmape(all_y_test, all_y_pred)
print(f'{"Mean":>6}  {mean_mae_saar/12:>8.1f}  {mean_smape:>8.2f}')
print()
print('Expected (Table 7.2): F1=8.2/14.00  F2=1.0/1.57  F3=1.6/2.68  F4=3.3/4.52  F5=11.2/13.40  Mean=5.1/7.23')

XGBoost CV complete. Verification (monthly equiv. = SAAR / 12):
  Fold    MAE_mo    sMAPE%
     1       8.2     14.00
     2       1.0      1.57
     3       1.6      2.68
     4       3.3      4.52
     5      11.2     13.40
  Mean       5.1      7.23

Expected (Table 7.2): F1=8.2/14.00  F2=1.0/1.57  F3=1.6/2.68  F4=3.3/4.52  F5=11.2/13.40  Mean=5.1/7.23


## Step 2 – Naive Random Walk CV

One-step naive predictor: $\hat{y}_t = y_{t-1}$.

Since the feature matrix already contains `fdefx_lag_1 = FDEFX.shift(1)`, the naive
prediction on the validation set is simply `X_test['fdefx_lag_1']` — no model is fitted.
This is identical to the carry-forward protocol used in the XGBoost evaluation: both
use the same validation indices and the same true historical lag values.

**MASE:** For each fold, the in-sample one-step naive MAE is computed on the training set
as `MAE(y_train, X_train['fdefx_lag_1'])`. MASE = XGBoost_MAE_fold / in_sample_naive_MAE_fold.
MASE < 1 indicates XGBoost beats the naive baseline for that fold.

In [5]:
naive_results = []

for r in xgb_results:
    y_test      = r['y_test']
    X_test_fold = X.iloc[r['test_idx']]
    X_train_fold = r['X_train']
    y_train      = r['y_train']

    # Naive one-step prediction: y_hat_t = y_{t-1} = fdefx_lag_1
    y_pred_naive = X_test_fold['fdefx_lag_1'].values

    mae_naive   = mean_absolute_error(y_test, y_pred_naive)
    rmse_naive  = np.sqrt(mean_squared_error(y_test, y_pred_naive))
    smape_naive = smape(y_test, y_pred_naive)
    wmape_naive = wmape(y_test, y_pred_naive)

    # In-sample one-step naive MAE (denominator for MASE)
    y_pred_insample_naive = X_train_fold['fdefx_lag_1'].values
    insample_naive_mae    = mean_absolute_error(y_train, y_pred_insample_naive)

    mase = r['MAE_saar'] / insample_naive_mae if insample_naive_mae > 0 else np.nan

    naive_results.append({
        'fold':              r['fold'],
        'test_idx':          r['test_idx'],
        'y_test':            y_test,
        'y_pred_naive':      y_pred_naive,
        'MAE_saar':          mae_naive,
        'RMSE_saar':         rmse_naive,
        'sMAPE':             smape_naive,
        'WMAPE':             wmape_naive,
        'insample_naive_mae': insample_naive_mae,
        'MASE':              mase,
    })

print('Naive RW CV complete.')
print(f'{"Fold":>6}  {"MAE_mo(Naive)":>14}  {"sMAPE%(Naive)":>14}  {"InSampleNaiveMAE":>17}  {"MASE":>6}')
for r in naive_results:
    print(f'{r["fold"]:>6}  {r["MAE_saar"]/12:>14.1f}  {r["sMAPE"]:>14.2f}  {r["insample_naive_mae"]/12:>17.1f}  {r["MASE"]:>6.3f}')

Naive RW CV complete.
  Fold   MAE_mo(Naive)   sMAPE%(Naive)   InSampleNaiveMAE    MASE
     1             0.4            0.64                0.3  25.401
     2             0.2            0.35                0.4   2.970
     3             0.2            0.32                0.3   5.198
     4             0.3            0.42                0.3  11.347
     5             0.5            0.56                0.3  38.753


## Step 3 – CV Comparison Table

In [6]:
rows = []
for xr, nr in zip(xgb_results, naive_results):
    rows.append({
        'Fold':         xr['fold'],
        # XGBoost (monthly equivalent)
        'XGB_MAE_mo':   round(xr['MAE_saar']  / 12, 1),
        'XGB_RMSE_mo':  round(xr['RMSE_saar'] / 12, 1),
        'XGB_sMAPE':    round(xr['sMAPE'],          2),
        'XGB_WMAPE':    round(xr['WMAPE'],           2),
        # Naive RW (monthly equivalent)
        'RW_MAE_mo':    round(nr['MAE_saar']  / 12, 1),
        'RW_RMSE_mo':   round(nr['RMSE_saar'] / 12, 1),
        'RW_sMAPE':     round(nr['sMAPE'],           2),
        'RW_WMAPE':     round(nr['WMAPE'],            2),
        # MASE (XGBoost vs in-sample naive)
        'MASE':         round(nr['MASE'],             3),
    })

compare_df = pd.DataFrame(rows).set_index('Fold')

# Mean row
mean_row = compare_df.mean().round(3)
# Recalculate sMAPE/WMAPE mean as simple mean (consistent with NB04)
mean_row['XGB_sMAPE'] = round(np.mean([xr['sMAPE'] for xr in xgb_results]), 2)
mean_row['XGB_WMAPE'] = round(np.mean([xr['WMAPE'] for xr in xgb_results]), 2)
mean_row['RW_sMAPE']  = round(np.mean([nr['sMAPE'] for nr in naive_results]), 2)
mean_row['RW_WMAPE']  = round(np.mean([nr['WMAPE'] for nr in naive_results]), 2)
mean_row.name = 'Mean'

compare_df = pd.concat([compare_df, mean_row.to_frame().T])
compare_df.index.name = 'Fold'

print('CV Comparison (MAE/RMSE in monthly USD bn = SAAR / 12):')
print(compare_df.to_string())

CV Comparison (MAE/RMSE in monthly USD bn = SAAR / 12):
      XGB_MAE_mo  XGB_RMSE_mo  XGB_sMAPE  XGB_WMAPE  RW_MAE_mo  RW_RMSE_mo  RW_sMAPE  RW_WMAPE    MASE
Fold                                                                                                  
1           8.20          9.6      14.00      13.41       0.40        0.70      0.64      0.63  25.401
2           1.00          1.3       1.57       1.56       0.20        0.50      0.35      0.35   2.970
3           1.60          1.9       2.68       2.65       0.20        0.40      0.32      0.32   5.198
4           3.30          4.1       4.52       4.51       0.30        0.70      0.42      0.41  11.347
5          11.20         13.1      13.40      12.85       0.50        1.00      0.56      0.56  38.753
Mean        5.06          6.0       7.23       7.00       0.32        0.66      0.46      0.45  16.734


## Step 4 – OOS Q1 2026

Three-way comparison: Naive anchored vs XGBoost forecast vs FDEFX actual.

- **Naive anchored:** last known FDEFX at Dec 2025 (Q4-2025 actual), carried forward.
- **XGBoost:** Q1 mean of `defense_forecast_2026_sac.csv` (Jan–Mar 2026).
- **Actual:** FDEFX Q1 2026 from FRED (loaded via fredapi); fallback if unavailable.

In [7]:
# Last known FDEFX at cutoff 31.12.2025 (Q4-2025 actual from feature matrix)
last_fdefx_saar = float(df['FDEFX'].iloc[-1])  # USD bn SAAR
print(f'Last FDEFX (Dec 2025): {last_fdefx_saar:.3f} USD bn SAAR')

# Naive anchored monthly prediction: carry Q4-2025 SAAR forward → divide by 12
naive_q1_monthly = last_fdefx_saar / 12
print(f'Naive anchored Q1 monthly: {naive_q1_monthly:.3f} USD bn/month')

# XGBoost Q1 2026: mean of Jan/Feb/Mar monthly values from SAC export
sac_df = pd.read_csv(DATA_PROCESSED + 'defense_forecast_2026_sac.csv')
sac_df['Date'] = sac_df['Date'].astype(str)
sac_q1 = sac_df[sac_df['Date'].isin(['202601', '202602', '202603'])]
xgb_q1_monthly = sac_q1['Net_Value_USD'].mean() / 1e9  # convert to USD bn
print(f'XGBoost Q1 monthly mean: {xgb_q1_monthly:.3f} USD bn/month')

Last FDEFX (Dec 2025): 1159.184 USD bn SAAR
Naive anchored Q1 monthly: 96.599 USD bn/month
XGBoost Q1 monthly mean: 75.472 USD bn/month


In [8]:
# Attempt to load Q1 2026 FDEFX actual from FRED.
# FDEFX is quarterly; Q1 2026 = observation dated 2026-01-01.
# Fallback values if FRED is unavailable: Q4 2025 = 1159.2, Q1 2026 = 1170.6 USD bn SAAR.
FALLBACK_Q4_2025_SAAR = 1159.2
FALLBACK_Q1_2026_SAAR = 1170.6

actual_q1_saar   = None
actual_source    = None

try:
    from dotenv import load_dotenv
    load_dotenv()
    api_key = os.getenv('FRED_API_KEY')
    if api_key is None:
        raise EnvironmentError('FRED_API_KEY not set in .env')

    from fredapi import Fred
    fred = Fred(api_key=api_key)
    fdefx_series = fred.get_series('FDEFX', observation_start='2025-10-01', observation_end='2026-06-30')

    # Check if Q1 2026 observation exists (dated 2026-01-01)
    q1_obs = fdefx_series.get(pd.Timestamp('2026-01-01'))
    if q1_obs is not None and not np.isnan(q1_obs):
        actual_q1_saar = float(q1_obs)
        actual_source  = 'FRED (live)'
    else:
        actual_q1_saar = FALLBACK_Q1_2026_SAAR
        actual_source  = 'fallback (FRED Q1 2026 not yet released)'
except Exception as e:
    actual_q1_saar = FALLBACK_Q1_2026_SAAR
    actual_source  = f'fallback ({e})'

actual_q1_monthly = actual_q1_saar / 12
print(f'Actual Q1 2026 FDEFX: {actual_q1_saar:.1f} USD bn SAAR  (source: {actual_source})')
print(f'Actual Q1 monthly:    {actual_q1_monthly:.3f} USD bn/month')

Actual Q1 2026 FDEFX: 1170.6 USD bn SAAR  (source: fallback (FRED_API_KEY not set in .env))
Actual Q1 monthly:    97.550 USD bn/month


In [9]:
# Absolute percentage errors vs actual Q1 2026
def ape(pred, actual):
    """Absolute percentage error (%)."""
    return abs(pred - actual) / abs(actual) * 100

naive_ape = ape(naive_q1_monthly, actual_q1_monthly)
xgb_ape   = ape(xgb_q1_monthly,   actual_q1_monthly)

print('OOS Q1 2026 Comparison (monthly USD bn = SAAR / 12):')
print(f'{"Model":<25}  {"Monthly USD bn":>14}  {"APE vs Actual":>14}')
print('-' * 58)
print(f'{"Naive (carry-forward)":<25}  {naive_q1_monthly:>14.3f}  {naive_ape:>13.1f}%')
print(f'{"XGBoost":<25}  {xgb_q1_monthly:>14.3f}  {xgb_ape:>13.1f}%')
print(f'{"Actual (FDEFX Q1 2026)":<25}  {actual_q1_monthly:>14.3f}  {0.0:>13.1f}%')
print(f'\nActual source: {actual_source}')

OOS Q1 2026 Comparison (monthly USD bn = SAAR / 12):
Model                      Monthly USD bn   APE vs Actual
----------------------------------------------------------
Naive (carry-forward)              96.599            1.0%
XGBoost                            75.472           22.6%
Actual (FDEFX Q1 2026)             97.550            0.0%

Actual source: fallback (FRED_API_KEY not set in .env)


## Step 5 – Export CSV

In [10]:
# Export CV comparison table
cv_out = compare_df.copy()
cv_out.index.name = 'Fold'
cv_path = DATA_PROCESSED + 'naive_baseline_cv_comparison.csv'
cv_out.to_csv(cv_path)
print(f'CV comparison saved: {cv_path}')

# Export OOS comparison
oos_rows = [
    {'Model': 'Naive (carry-forward)', 'Monthly_USD_bn': round(naive_q1_monthly, 3), 'APE_pct': round(naive_ape, 1)},
    {'Model': 'XGBoost',               'Monthly_USD_bn': round(xgb_q1_monthly,   3), 'APE_pct': round(xgb_ape,   1)},
    {'Model': f'Actual ({actual_source})', 'Monthly_USD_bn': round(actual_q1_monthly, 3), 'APE_pct': 0.0},
]
oos_df   = pd.DataFrame(oos_rows)
oos_path = DATA_PROCESSED + 'naive_baseline_oos_q1_2026.csv'
oos_df.to_csv(oos_path, index=False)
print(f'OOS comparison saved: {oos_path}')

CV comparison saved: ../data/processed/naive_baseline_cv_comparison.csv


OOS comparison saved: ../data/processed/naive_baseline_oos_q1_2026.csv


## Step 6 – LaTeX Output

In [11]:
# Build Table 7.2 extended with Naive RW and MASE columns.
# MAE / RMSE reported in monthly USD bn (SAAR / 12).
# All values taken from compare_df (already rounded).

latex_lines = [
    r'% ----- Table 7.2 extended: XGBoost vs. Naive Random Walk -----',
    r'\begin{table}[htbp]',
    r'  \centering',
    r'  \caption{Cross-Validation Results: XGBoost vs.\ Naive Random Walk (FDEFX, US Defense Market).'
    r' MAE and RMSE in monthly USD~bn (SAAR~$\div$~12). sMAPE and WMAPE in~\%.}',
    r'  \label{tab:cv_xgb_naive}',
    r'  \begin{tabular}{lrrrrrrrrrr}',
    r'    \toprule',
    r'    & \multicolumn{4}{c}{XGBoost} & \multicolumn{4}{c}{Naive (RW)} & \\',
    r'    \cmidrule(lr){2-5}\cmidrule(lr){6-9}',
    r'    Fold & MAE & RMSE & sMAPE & WMAPE & MAE & RMSE & sMAPE & WMAPE & MASE \\',
    r'    \midrule',
]

fold_labels = list(compare_df.index)
for label in fold_labels:
    row = compare_df.loc[label]
    if label == 'Mean':
        latex_lines.append(r'    \midrule')
    latex_lines.append(
        f'    {label} & '
        f'{row["XGB_MAE_mo"]:.1f} & '
        f'{row["XGB_RMSE_mo"]:.1f} & '
        f'{row["XGB_sMAPE"]:.2f} & '
        f'{row["XGB_WMAPE"]:.2f} & '
        f'{row["RW_MAE_mo"]:.1f} & '
        f'{row["RW_RMSE_mo"]:.1f} & '
        f'{row["RW_sMAPE"]:.2f} & '
        f'{row["RW_WMAPE"]:.2f} & '
        f'{row["MASE"]:.3f} \\\\'
    )

latex_lines += [
    r'    \bottomrule',
    r'  \end{tabular}',
    r'\end{table}',
    '',
    r'% ----- OOS Table: Q1 2026 Comparison -----',
    r'\begin{table}[htbp]',
    r'  \centering',
    r'  \caption{Out-of-Sample Comparison Q1~2026: Naive vs.\ XGBoost vs.\ Actual FDEFX.'
    r' Monthly market volume in USD~bn (FDEFX SAAR~$\div$~12).}',
    r'  \label{tab:oos_q1_2026}',
    r'  \begin{tabular}{lrr}',
    r'    \toprule',
    r'    Model & Monthly USD bn & APE (\%) \\',
    r'    \midrule',
    f'    Naive (carry-forward) & {naive_q1_monthly:.1f} & {naive_ape:.1f} \\\\',
    f'    XGBoost               & {xgb_q1_monthly:.1f} & {xgb_ape:.1f} \\\\',
    f'    Actual (FDEFX Q1~2026) & {actual_q1_monthly:.1f} & 0.0 \\\\',
    r'    \bottomrule',
    r'  \end{tabular}',
    r'\end{table}',
]

print('\n'.join(latex_lines))

% ----- Table 7.2 extended: XGBoost vs. Naive Random Walk -----
\begin{table}[htbp]
  \centering
  \caption{Cross-Validation Results: XGBoost vs.\ Naive Random Walk (FDEFX, US Defense Market). MAE and RMSE in monthly USD~bn (SAAR~$\div$~12). sMAPE and WMAPE in~\%.}
  \label{tab:cv_xgb_naive}
  \begin{tabular}{lrrrrrrrrrr}
    \toprule
    & \multicolumn{4}{c}{XGBoost} & \multicolumn{4}{c}{Naive (RW)} & \\
    \cmidrule(lr){2-5}\cmidrule(lr){6-9}
    Fold & MAE & RMSE & sMAPE & WMAPE & MAE & RMSE & sMAPE & WMAPE & MASE \\
    \midrule
    1 & 8.2 & 9.6 & 14.00 & 13.41 & 0.4 & 0.7 & 0.64 & 0.63 & 25.401 \\
    2 & 1.0 & 1.3 & 1.57 & 1.56 & 0.2 & 0.5 & 0.35 & 0.35 & 2.970 \\
    3 & 1.6 & 1.9 & 2.68 & 2.65 & 0.2 & 0.4 & 0.32 & 0.32 & 5.198 \\
    4 & 3.3 & 4.1 & 4.52 & 4.51 & 0.3 & 0.7 & 0.42 & 0.41 & 11.347 \\
    5 & 11.2 & 13.1 & 13.40 & 12.85 & 0.5 & 1.0 & 0.56 & 0.56 & 38.753 \\
    \midrule
    Mean & 5.1 & 6.0 & 7.23 & 7.00 & 0.3 & 0.7 & 0.46 & 0.45 & 16.734 \\
    \bottomrule
  \e

## Summary

**In-distribution (MASE verdict):** MASE exceeds 1 in every fold (range 2.97–38.75, mean 16.73),
meaning XGBoost is consistently worse than the naive carry-forward on the cross-validation
validation sets. This result is structural rather than a model failure: FDEFX is quarterly
forward-filled, so the carry-forward predictor has zero error for approximately two-thirds of
all months (the within-quarter months where $y_t = y_{t-1}$). XGBoost accumulates errors even
on these months due to the mean-reversion bias documented in NB04 (Fold-5 structural break).
The sMAPE contrast (XGBoost mean 7.23 % vs. Naive 0.46 %) confirms this gap.
The high MASE should be disclosed as a methodological limitation: the naive benchmark
exploits the FDEFX data-generating process (quarterly forward-fill) in a way that an ML model
cannot match without essentially also being a carry-forward. For a genuinely monthly target
variable the comparison would be more informative.

**OOS Q1 2026:** The naive carry-forward (96.6 USD bn/month, APE 1.0 %) decisively outperforms
XGBoost (75.5 USD bn/month, APE 22.6 %) against the Q1 2026 actual (97.5 USD bn/month,
fallback). The XGBoost mean-reversion to the historical training mean (~900 USD bn SAAR)
materially underestimates the sustained post-2022 elevated defense spending regime, which the
naive anchor captures by construction.